# TAAS 추가 데이터 탐색

`data2/` 폴더의 TAAS 엑셀 파일을 살펴보고, 연도별 추세·가해운전자 연령대·법규위반·인구 10만 명당 사고를 비교합니다.

먼저 파일과 시트의 미리보기 및 열 정보를 확인하세요. 원본 파일마다 표 구조가 다를 수 있으므로, 아래 분석 셀에서 사용할 시트와 열 이름을 확인한 뒤 설정값을 지정합니다. 원본 파일은 수정하지 않습니다.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("../data2")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"데이터 폴더를 찾을 수 없습니다: {DATA_DIR.resolve()}")

plt.rcParams["figure.figsize"] = (11, 5)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
files = sorted(DATA_DIR.glob("*.xlsx"))
if not files:
    raise FileNotFoundError(f"{DATA_DIR.resolve()}에서 .xlsx 파일을 찾을 수 없습니다.")

print(f"엑셀 파일 {len(files)}개")
for path in files:
    print(f"- {path.name} ({path.stat().st_size:,} bytes)")

## 1. 파일·시트 구조 확인

각 시트의 원본 상단 행을 먼저 표시합니다. 제목 행이나 병합 셀이 있는지 살펴보고, 그 다음 열 이름을 지정해 읽은 표의 구조도 확인합니다.

In [ ]:
workbooks = {}
sheet_catalog = []

for path in files:
    excel = pd.ExcelFile(path)
    workbooks[path.name] = {}
    print(f"\n{'=' * 90}\n파일: {path.name}\n시트: {excel.sheet_names}")
    for sheet_name in excel.sheet_names:
        raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
        table = pd.read_excel(path, sheet_name=sheet_name)
        workbooks[path.name][sheet_name] = table
        sheet_catalog.append({
            "파일": path.name,
            "시트": sheet_name,
            "행 수": len(table),
            "열 수": len(table.columns),
            "열 이름 (header=0)": list(table.columns),
        })
        print(f"\n시트: {sheet_name!r} | 크기(header=0): {table.shape}")
        print("원본 상단 8행 (열 이름 미지정):")
        display(raw.head(8))
        print("열 이름 / 자료형:")
        display(pd.DataFrame({"열 이름": table.columns.astype(str), "자료형": table.dtypes.astype(str).values}))

catalog = pd.DataFrame(sheet_catalog)
display(catalog[["파일", "시트", "행 수", "열 수"]])

## 2. 결측치와 기초 통계

시트별 결측치와 수치형 변수의 요약 통계를 확인합니다. 값이 숫자로 저장되지 않은 열이 있으면 원본 미리보기에서 형식을 확인하세요.

In [ ]:
for filename, sheets in workbooks.items():
    for sheet_name, df in sheets.items():
        print(f"\n## {filename} / {sheet_name}")
        missing = df.isna().sum()
        print(f"행 {len(df):,}개, 열 {len(df.columns):,}개")
        display(missing.rename("결측치 수").to_frame().query("`결측치 수` > 0"))
        numeric = df.select_dtypes(include="number")
        if numeric.empty:
            print("수치형 열이 없습니다.")
        else:
            display(numeric.describe().T)

## 3. 비교 그래프

아래 도우미는 열 이름을 직접 지정해 그립니다. 각 관심 주제의 시트에서 `df.head()`와 `df.columns.tolist()`을 확인한 뒤 설정값을 채우세요. 숫자 열의 쉼표는 자동으로 처리합니다.

In [ ]:
def numeric_values(series):
    """쉼표가 포함된 문자열도 숫자로 변환합니다."""
    cleaned = series.astype("string").str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")


def plot_category(df, category_col, value_col, title, *, top_n=20, ascending=False):
    """범주별 수치를 막대그래프로 표시합니다."""
    missing = {category_col, value_col} - set(df.columns)
    if missing:
        raise KeyError(f"없는 열: {sorted(missing)}. 실제 열: {df.columns.tolist()}")
    data = df[[category_col, value_col]].copy()
    data[value_col] = numeric_values(data[value_col])
    data = data.dropna(subset=[category_col, value_col])
    grouped = data.groupby(category_col, as_index=False)[value_col].sum()
    grouped = grouped.sort_values(value_col, ascending=ascending).head(top_n)
    ax = sns.barplot(data=grouped, x=value_col, y=category_col, color="steelblue")
    ax.set(title=title, xlabel=value_col, ylabel=category_col)
    plt.tight_layout()
    plt.show()
    return grouped


def plot_year_columns(df, label_col, *, title, year_columns=None):
    """연도 열이 가로로 놓인 표에서 각 항목의 연도 추이를 표시합니다."""
    if label_col not in df.columns:
        raise KeyError(f"'{label_col}' 열이 없습니다. 실제 열: {df.columns.tolist()}")
    if year_columns is None:
        year_columns = [col for col in df.columns if re.search(r"(?:19|20)\d{2}", str(col))]
    if not year_columns:
        raise ValueError("연도 열을 찾지 못했습니다. year_columns에 연도 열 이름을 직접 지정하세요.")
    data = df[[label_col, *year_columns]].copy()
    long = data.melt(id_vars=label_col, var_name="연도", value_name="값")
    long["연도"] = long["연도"].astype(str).str.extract(r"((?:19|20)\d{2})")[0]
    long["값"] = numeric_values(long["값"])
    long = long.dropna(subset=["연도", "값"])
    plt.figure(figsize=(11, 5))
    sns.lineplot(data=long, x="연도", y="값", hue=label_col, marker="o")
    plt.xticks(rotation=45)
    plt.title(title)
    plt.tight_layout()
    plt.show()
    return long

### 3.1 1995–2025년 사고 추세

연도별 추세 파일에서 연도 열이 가로 방향인지 먼저 확인하세요. `항목열`에는 사고건수·사망자수·부상자수 등이 적힌 열 이름을 지정합니다. 여러 항목을 한 번에 그리면 항목마다 값의 규모 차이가 클 수 있습니다.

In [ ]:
# 아래 파일명과 시트 이름은 catalog 출력에서 확인해 지정하세요.
trend_file = next((name for name in workbooks if "추세" in name), None)
print("추세 파일 후보:", trend_file)
if trend_file:
    for sheet_name, trend_df in workbooks[trend_file].items():
        print(f"시트: {sheet_name!r} / 열: {trend_df.columns.tolist()}")
        display(trend_df.head())

# 예시: trend_df = workbooks[trend_file]["시트 이름"]
# 항목열 = "구분2"  # 실제 열 이름으로 바꾸기
# plot_year_columns(trend_df, 항목열, title="연도별 교통사고 추이")

### 3.2 가해운전자 연령대별 사고

연령대 열과 비교할 사고 지표 열을 확인한 뒤 설정합니다. 자료가 연령대별 행 형태라면 아래 막대그래프를 사용할 수 있습니다.

In [ ]:
age_file = next((name for name in workbooks if "연령" in name), None)
print("연령 파일 후보:", age_file)
if age_file:
    age_sheet = next(iter(workbooks[age_file]))
    age_df = workbooks[age_file][age_sheet]
    print("시트:", age_sheet, "| 열:", age_df.columns.tolist())
    display(age_df.head(10))

# 아래 두 열을 실제 열 이름으로 바꿔 실행하세요.
# 연령열 = "연령대"
# 사고건수열 = "사고건수"
# plot_category(age_df, 연령열, 사고건수열, "가해운전자 연령대별 사고건수")

### 3.3 법규위반 유형별 사고

법규위반 유형을 범주 열로, 비교할 사고 지표를 값 열로 지정하세요. 상위 20개 범주를 사고건수 순으로 표시합니다.

In [ ]:
violation_file = next((name for name in workbooks if "법규" in name), None)
print("법규위반 파일 후보:", violation_file)
if violation_file:
    violation_sheet = next(iter(workbooks[violation_file]))
    violation_df = workbooks[violation_file][violation_sheet]
    print("시트:", violation_sheet, "| 열:", violation_df.columns.tolist())
    display(violation_df.head(10))

# 아래 두 열을 실제 열 이름으로 바꿔 실행하세요.
# 위반유형열 = "법규위반"
# 사고건수열 = "사고건수"
# plot_category(violation_df, 위반유형열, 사고건수열, "법규위반 유형별 사고건수")

### 3.4 인구 10만 명당 사고

인구 대비 사고 지표는 원자료에 계산된 비율이 있으면 그 열을 사용합니다. 지역별 비교 자료에서 비율 열이 없는 경우에는 사고건수와 인구 열로 직접 계산할 수 있습니다. 열 이름과 단위를 확인하고 아래 식을 적용하세요.

In [ ]:
population_file = next((name for name in workbooks if "인구10만명당" in name), None)
print("인구 10만 명당 파일 후보:", population_file)
if population_file:
    population_sheet = next(iter(workbooks[population_file]))
    population_df = workbooks[population_file][population_sheet]
    print("시트:", population_sheet, "| 열:", population_df.columns.tolist())
    display(population_df.head(10))

# 원자료에 이미 있는 '10만명당 사고' 열을 사용하거나,
# 사고건수와 인구 수가 각각 있는 경우 다음처럼 계산합니다.
# 인구대비_df = population_df.copy()
# 인구대비_df["10만명당 사고"] = (
#     numeric_values(인구대비_df[사고건수열])
#     / numeric_values(인구대비_df[인구수열]) * 100_000
# )
# plot_category(인구대비_df, 지역열, "10만명당 사고", "지역별 인구 10만 명당 사고")

## 탐색 질문

- 1995년 이후 사고건수와 사망자수는 같은 방향으로 변화했나요?
- 연령대별 사고건수와 인구 10만 명당 사고 순위는 어떻게 다른가요?
- 사고가 많이 발생한 법규위반 유형은 무엇인가요?
- 단순 건수와 인구 대비 비율로 비교했을 때 지역 간 해석이 어떻게 달라지나요?

범주별 건수를 비교하기 전에 연도, 지역, 단위가 동일한 행끼리 비교하는지 확인하고, 원자료의 주석과 정의도 함께 살펴보세요.